<a href="https://colab.research.google.com/github/Ameena1BM23CS27/6thSem-ML-Lab/blob/main/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv("heart (1).csv")
print(df.head())


   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  


In [4]:

categorical_cols = df.select_dtypes(include=['object']).columns

# Apply Label Encoding (for binary categories)
le = LabelEncoder()
for col in categorical_cols:
    if df[col].nunique() == 2:
        df[col] = le.fit_transform(df[col])

# Apply One-Hot Encoding (for multi-category)
df = pd.get_dummies(df, drop_first=True)

In [6]:
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [8]:
models = {
    "SVM": SVC(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42)
}

print("\n--- Accuracy WITHOUT PCA ---")
results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)

    results[name] = acc
    print(f"{name}: {acc:.4f}")

# Best model without PCA
best_model_name = max(results, key=results.get)
print(f"\nBest Model (Without PCA): {best_model_name}")


--- Accuracy WITHOUT PCA ---
SVM: 0.8750
Logistic Regression: 0.8533
Random Forest: 0.8587

Best Model (Without PCA): SVM


In [9]:
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"\nReduced dimensions: {X_train_pca.shape[1]}")


Reduced dimensions: 13


In [10]:
print("\n--- Accuracy WITH PCA ---")
pca_results = {}

for name, model in models.items():
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)
    acc = accuracy_score(y_test, y_pred)

    pca_results[name] = acc
    print(f"{name}: {acc:.4f}")

# Best model with PCA
best_pca_model = max(pca_results, key=pca_results.get)
print(f"\nBest Model (With PCA): {best_pca_model}")



--- Accuracy WITH PCA ---
SVM: 0.8804
Logistic Regression: 0.8533
Random Forest: 0.8641

Best Model (With PCA): SVM


In [11]:
print("\n--- Comparison ---")
for model in models.keys():
    print(f"{model}: Without PCA = {results[model]:.4f}, With PCA = {pca_results[model]:.4f}")


--- Comparison ---
SVM: Without PCA = 0.8750, With PCA = 0.8804
Logistic Regression: Without PCA = 0.8533, With PCA = 0.8533
Random Forest: Without PCA = 0.8587, With PCA = 0.8641
